# ARTICLE2_NB5: Figures and final tables

**This notebook produces results for the fault-injection study (see README).**

It reads only the saved outputs of NB1-NB4 (no model is trained, no simulation is re-run) and writes the manuscript figures and tables.

- **CELL 2 - Frozen diagnostic result.** Rebuilds the detection matrix under the original protocol scope (C2 on ARFIMA, LSTM and XGBoost only; thresholds unchanged; deviation log 12 Nov 2026) and **stops unless** the reference passes all seven checks and every fault is detected by at least one check. The expanded matrix (C2 also on the literature baselines) is kept as a supplementary table.
- **CELL 3 - Tables.** Main and supplementary tables as CSV.
- **CELLS 4-9 - Figures.** PNG (300 dpi) and PDF.

**Wording rule for the paper:** the matrix shows *detection coverage in this experiment* (one pipeline, one seed, faults injected one at a time). It is not an estimate of general detection reliability.

**Inputs and outputs:** reads only `Article 2/05_Outputs`; writes only `Article 2/05_Outputs/figures` and `Article 2/05_Outputs/tables`.
**Runtime:** under a minute; a GPU is not needed.

## CELL 0: Environment setup (Article 2 folder only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

%cd "/content/drive/MyDrive/Article 2"

import os, sys
from pathlib import Path
ROOT = Path(os.getcwd())
assert ROOT.name == "Article 2", f"Wrong folder: {ROOT}"
print("Working directory:", ROOT)

## CELL 1: Imports, style and inputs

In [ ]:
import json, pickle
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy import stats

OUT = ROOT / "05_Outputs"
FIG, TAB = OUT / "figures", OUT / "tables"
for d in [FIG, TAB]:
    d.mkdir(parents=True, exist_ok=True)
TICKERS = ["JPM", "NEE", "XOM"]
MODELS = ["ARFIMA", "LSTM", "XGBoost", "Serial (ARFIMA+LSTM)", "Serial (Final)",
          "Parallel Equal", "Parallel InvError", "Parallel SoftMax", "Stacking"]
ARCH = ["Serial (Final)", "Parallel Equal", "Parallel InvError", "Parallel SoftMax", "Stacking"]
BASELINES = ["ARIMA", "ARIMA -> LSTM", "ARIMA -> XGBoost", "XGBoost -> LSTM", "ARFIMA -> LSTM", "Attention-CNN-LSTM", "AttCLX"]
FAULTS = ["F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8"]
FAULT_NAMES = {"F1": "Silent estimation failure", "F2": "Horizon mismatch", "F3": "Unit mismatch", "F4": "Target leakage",
               "F5": "Placeholder component", "F6": "Pseudo-residual cascade", "F7": "Test misspecification",
               "F8": "Non-identical baselines"}
CHECKS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7"]
EXPECTED = {"C1": ["F1"], "C2": ["F4"], "C3": ["F3"], "C4": ["F5", "F6"], "C5": ["F6"], "C6": ["F2", "F4"], "C7": ["F7", "F8"]}
PRIMARY_COMPONENTS = ["ARFIMA", "LSTM", "XGBoost"]

need = ["faults/faults_delta_summary.csv", "faults/faults_apparent_improvement_summary.csv", "faults/faults_significance_and_tau.csv",
        "faults/reference_metrics_long.csv", "tests/F7_summary_by_seed.csv", "tests/_mc_raw.pkl", "baselines/F8_delta_summary.csv",
        "baselines/baseline_and_research_metrics_long.csv", "baselines/F8_ranking_by_seed.csv", "combined/combined_metrics_long.csv",
        "combined/combined_dm_sets.csv", "combined/combined_f7.csv", "combined/v1_claims_vs_combined.csv",
        "diagnostics/check_reasons.csv", "diagnostics/detection_matrix.csv", "diagnostics/f7_84_three_ways.csv",
        "diagnostics/reference_counts_holm_vs_unadjusted.csv", "reference/reference_seed_range.csv"]
missing = [p for p in need if not (OUT / p).exists()]
assert not missing, f"Run NB1-NB4 first. Missing: {missing}"

# Style (validated categorical slots 1-3; sequential blue; recessive chrome)
INK, INK2, GRID, NEUTRAL = "#0b0b0b", "#52514e", "#e6e5e0", "#f0efec"
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
BLUE = {100: "#cde2fb", 250: "#86b6ef", 400: "#3987e5", 550: "#1c5cab", 700: "#0d366b"}
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#184f95", "#86b6ef", NEUTRAL, "#f1a09f", "#e34948"])
matplotlib.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "legend.frameon": False, "figure.dpi": 120,
    "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42})

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", facecolor="white")
    fig.savefig(FIG / f"{name}.pdf", facecolor="white")
    plt.show()
    print("saved", name)

read = lambda p: pd.read_csv(OUT / p)
print("Inputs found")

## CELL 2: Frozen diagnostic result (original protocol scope)

C2 is re-evaluated on the primary forecasting components only (ARFIMA, LSTM, XGBoost), from the saved per-component training-fit tables, with the frozen thresholds (training returns R2 above 0.20 or training MAE ratio below 0.80). All other checks are taken from NB4 unchanged.

In [ ]:
reasons = read("diagnostics/check_reasons.csv")
runs = list(dict.fromkeys(reasons.Run))
fail = {(r.Run, r.Check): r.Result == "FAIL" for r in reasons.itertuples()}

c2_rows = []
for run in runs:
    c2 = pd.read_csv(OUT / "diagnostics" / f"C2_training_fit_{run}.csv")
    prim = c2[c2.Component.isin(PRIMARY_COMPONENTS)]
    bad = prim[(prim.Train_returns_R2 > 0.20) | (prim.Train_MAE_ratio < 0.80)]
    fail[(run, "C2")] = not bad.empty
    c2_rows.append({"Run": run, "C2_primary_scope": "FAIL" if not bad.empty else "pass",
                    "Reason": "; ".join(f"{r.Ticker} {r.Component}: R2 {r.Train_returns_R2:.3f}, ratio {r.Train_MAE_ratio:.3f}"
                                        for r in bad.itertuples()) or "training fit plausible"})
c2_primary = pd.DataFrame(c2_rows)

def mark(run, c):
    f = fail[(run, c)]
    if run in EXPECTED[c]:
        return "FAIL (expected)" if f else "pass (MISSED)"
    return "FAIL" if f else "pass"

matrix = pd.DataFrame({c: [mark(r, c) for r in runs] for c in CHECKS}, index=pd.Index(runs, name="Run"))
detected_by = {f: [c for c in CHECKS if fail[(f, c)]] for f in FAULTS}
REF_PASSES_ALL = not any(fail[("REF", c)] for c in CHECKS)
ALL_DETECTED = all(len(v) > 0 for v in detected_by.values())
SINGLE_LINE = [f for f in FAULTS if len(detected_by[f]) == 1]
MISSED = [(c, f) for c in CHECKS for f in EXPECTED[c] if not fail[(f, c)]]

print("DETECTION MATRIX - original protocol scope (seed S0)")
print(matrix.to_string())
print()
print("C2 on the primary components:")
print(c2_primary.to_string(index=False))
print()
print("Detected by:", {f: ", ".join(v) for f, v in detected_by.items()})
print("Single line of defence:", {f: detected_by[f][0] for f in SINGLE_LINE})
print("Reference passes all seven checks:", REF_PASSES_ALL)
print("Every fault detected by at least one check:", ALL_DETECTED)
print("Expected detections missed:", MISSED or "none")
assert REF_PASSES_ALL and ALL_DETECTED, "Does not match the result frozen on 12 Nov 2026. Stop and report."
print()
print("Matches the frozen result. Coverage in this experiment only; not a general reliability estimate.")

matrix.to_csv(TAB / "Table_5_detection_matrix_primary_scope.csv")
pd.read_csv(OUT / "diagnostics" / "detection_matrix.csv").to_csv(TAB / "Table_S4_detection_matrix_C2_with_baselines.csv", index=False)
c2_base = pd.concat([pd.read_csv(OUT / "diagnostics" / f"C2_training_fit_{r}.csv") for r in ["REF", "F8"]])
c2_base = c2_base[~c2_base.Component.isin(PRIMARY_COMPONENTS)].round(3)
c2_base.to_csv(TAB / "Table_S5_baseline_training_fit_overfitting.csv", index=False)
print()
print("Baseline training fit (supplementary; REF = fair baselines, F8 = v1 route):")
print(c2_base.pivot_table(index="Component", columns=["Run", "Ticker"], values="Train_returns_R2").round(3).to_string())

## CELL 3: Tables (CSV)

In [ ]:
seed_range = read("reference/reference_seed_range.csv")
ref_m = read("faults/reference_metrics_long.csv")
d = read("faults/faults_delta_summary.csv")
comb_m = read("combined/combined_metrics_long.csv")

# Combined run: delta in the same format as the single faults
cm = comb_m.merge(ref_m[["Seed", "Ticker", "Model", "MAE_ratio"]], on=["Seed", "Ticker", "Model"], suffixes=("", "_ref"))
cm = cm[cm.Model != "Persistence"].assign(Delta=lambda x: x.MAE_ratio - x.MAE_ratio_ref)
cm_mean = cm.groupby(["Seed", "Model"])[["MAE_ratio", "Delta"]].mean().reset_index().assign(Ticker="Mean")
cm = pd.concat([cm, cm_mean])
comb_sum = cm.groupby(["Model", "Ticker"]).agg(Fault_ratio=("MAE_ratio", "mean"), Delta_mean=("Delta", "mean"),
                                               Delta_min=("Delta", "min"), Delta_max=("Delta", "max")).reset_index().assign(Fault="Combined")
dd = pd.concat([d, comb_sum], ignore_index=True)
fmt3 = lambda r: f"{r.Delta_mean:+.3f} [{r.Delta_min:+.3f}, {r.Delta_max:+.3f}]"
t2 = dd[dd.Ticker == "Mean"].assign(Cell=lambda x: x.apply(fmt3, axis=1)).pivot(index="Model", columns="Fault", values="Cell")
t2 = t2.loc[MODELS, ["F1", "F2", "F3", "F4", "F5", "F6", "Combined"]]
t2.to_csv(TAB / "Table_2_delta_MAE_ratio_stock_mean.csv")
dd.round(4).to_csv(TAB / "Table_S1_delta_MAE_ratio_per_stock.csv", index=False)
print("Table 2 - Delta MAE ratio, stock mean, mean over five seeds [min, max]")
print(t2.to_string())

# Significance counts
sig = read("faults/faults_significance_and_tau.csv").groupby("Fault")[["Sig_primary", "Sig_vs_persistence", "Sig_vs_ARFIMA", "Tau"]].mean()
cd = read("combined/combined_dm_sets.csv").groupby(["Seed", "Set"]).Sig_Holm.sum().unstack().mean()
sig.loc["Combined"] = [cd["primary"], cd["vs_persistence"], cd["vs_ARFIMA"], np.nan]
refc = read("faults/faults_significance_and_tau.csv")[["Ref_primary", "Ref_vs_persistence", "Ref_vs_ARFIMA"]].mean()
sig.loc["Reference"] = [refc.Ref_primary, refc.Ref_vs_persistence, refc.Ref_vs_ARFIMA, 1.0]
f7 = read("tests/F7_summary_by_seed.csv")
t3 = sig.round(2)
t3.to_csv(TAB / "Table_3_significance_counts_holm.csv")
three = read("diagnostics/f7_84_three_ways.csv")
three.to_csv(TAB / "Table_3b_84_pairs_three_ways.csv", index=False)
print()
print("Table 3 - significant DM results after Holm, mean over seeds (9 / 27 / 15 tests)")
print(t3.to_string())
print("F7 (84 plain tests, no correction): significant", f7.F7_significant_of_84.tolist())

# Monte Carlo, re-presented from the saved counts (mean, median and range for single tests)
with open(OUT / "tests" / "_mc_raw.pkl", "rb") as f:
    mc_raw = pickle.load(f)
reps = int(json.load(open(OUT / "logs" / "ARTICLE2_NB3_run_log.json"))["mc_reps"])
def ci(x, n):
    c = stats.binomtest(int(x), int(n)).proportion_ci(confidence_level=0.95, method="exact")
    return c.low, c.high
rows = []
for (n, block), r in sorted(mc_raw.items()):
    for name, cnt in [("(a) Plain DM, single test", r["rej_plain"]), ("(b) NW + HLN DM, single test", r["rej_hln"])]:
        rate = cnt / reps
        lo, hi = ci(int(round(rate.mean() * reps)), reps)
        rows.append({"n": n, "Mean_block": block, "Procedure": name, "Rate_mean": rate.mean(), "Rate_median": np.median(rate),
                     "Rate_min": rate.min(), "Rate_max": rate.max(), "CI95_low": lo, "CI95_high": hi})
    for name, x in [("(c) 84 plain tests, no correction", r["fam_plain"]), ("(d) 84 NW + HLN tests, Holm", r["fam_holm84"]),
                    ("Context: 9 primary NW + HLN tests, Holm", r["fam_holm9"])]:
        lo, hi = ci(x, reps)
        rows.append({"n": n, "Mean_block": block, "Procedure": name, "Rate_mean": x / reps, "Rate_median": np.nan,
                     "Rate_min": np.nan, "Rate_max": np.nan, "CI95_low": lo, "CI95_high": hi})
mc = pd.DataFrame(rows)
mc.round(3).to_csv(TAB / "Table_4_monte_carlo_test_size.csv", index=False)
print()
print(f"Table 4 - Monte Carlo ({reps} replications per setting). Single tests: mean rate over the 84 tests (interval for that mean treated as one proportion), median and range")
print(mc[mc.Mean_block == 10].round(3).to_string(index=False))

# F8 and v1 claims
read("baselines/F8_delta_summary.csv").round(4).to_csv(TAB / "Table_6_F8_baselines.csv", index=False)
read("combined/v1_claims_vs_combined.csv").to_csv(TAB / "Table_S4_v1_claims_vs_combined.csv", index=False)
seed_range.round(4).to_csv(TAB / "Table_S3_reference_seed_range.csv", index=False)
print()
print("Tables written to", TAB)

## CELL 4: Figure 1 - how far each fault moves the MAE ratio (stock mean, mean over seeds)

In [ ]:
rows = ["F1", "F2", "F3", "F4", "F5", "F6", "Combined"]
fig, ax = plt.subplots(figsize=(6.8, 3.4))
lo, hi = seed_range[seed_range.Stock == "Mean"]["min"].min(), seed_range[seed_range.Stock == "Mean"]["max"].max()
ax.axvspan(lo, hi, color=NEUTRAL, zorder=0)
ax.axvline(1, color=INK2, lw=0.8, zorder=1)
for i, f in enumerate(rows):
    sub = dd[(dd.Fault == f) & (dd.Ticker == "Mean")].set_index("Model").loc[MODELS]
    y = len(rows) - 1 - i
    ax.scatter(sub.Fault_ratio, np.full(len(sub), y), s=26, color=S1, edgecolor="white", linewidth=0.8, zorder=3)
    for side in ["max", "min"]:
        m = sub.Fault_ratio.idxmax() if side == "max" else sub.Fault_ratio.idxmin()
        v = sub.Fault_ratio[m]
        if abs(np.log10(v)) > 0.05:
            ax.annotate(f"{m} {v:,.0f}" if v >= 100 else f"{m} {v:.3g}", (v, y), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=6.5, color=INK2)
ax.set_xscale("log")
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([f"{f} {FAULT_NAMES[f]}" if f in FAULT_NAMES else "All eight faults combined" for f in rows][::-1])
ax.set_xlabel("MAE ratio to persistence (log scale; 1 = persistence)")
ax.set_title("Each dot is one model; grey band = reference range over five seeds", loc="left", color=INK2, fontsize=8)
ax.xaxis.grid(True, color=GRID, lw=0.5, which="major")
ax.set_ylim(-0.6, len(rows) - 0.3)
save(fig, "Figure_1_fault_effect_MAE_ratio")

## CELL 5: Figure 2 - apparent improvement over ARFIMA (fault minus reference, percentage points)

In [ ]:
app = read("faults/faults_apparent_improvement_summary.csv")
grid = app.pivot(index="Architecture", columns="Fault", values="Apparent_vs_ARFIMA_pp")
# combined run, same definition
def gains(m):
    p = m.pivot_table(index=["Seed", "Ticker"], columns="Model", values="MAE").reset_index()
    return pd.DataFrame({a: 100 * (p["ARFIMA"] - p[a]) / p["ARFIMA"] for a in ARCH})
grid["Combined"] = (gains(comb_m) - gains(ref_m)).mean().reindex(grid.index)
grid = grid.loc[ARCH, ["F1", "F2", "F3", "F4", "F5", "F6", "Combined"]]
grid.round(2).to_csv(TAB / "Table_S6_apparent_improvement_vs_ARFIMA_pp.csv")
fig, ax = plt.subplots(figsize=(6.8, 2.6))
lim = float(np.nanmax(np.abs(grid.values)))
im = ax.imshow(grid.values, cmap=DIVERGING, vmin=-lim, vmax=lim, aspect="auto")
for (i, j), v in np.ndenumerate(grid.values):
    ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=7, color="white" if abs(v) > 0.6 * lim else INK)
ax.set_xticks(range(grid.shape[1]), grid.columns)
ax.set_yticks(range(grid.shape[0]), grid.index)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xticks(np.arange(-0.5, grid.shape[1]), minor=True); ax.set_yticks(np.arange(-0.5, grid.shape[0]), minor=True)
ax.grid(which="minor", color="white", lw=2); ax.tick_params(which="minor", length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.set_label("percentage points", fontsize=7); cb.outline.set_visible(False)
ax.set_title("Gain over ARFIMA a researcher would report, minus the true gain (mean over seeds and stocks)", loc="left", color=INK2, fontsize=8)
save(fig, "Figure_2_apparent_improvement")

## CELL 6: Figure 3 - Monte Carlo: probability of naming at least one false winner

In [ ]:
procs = ["(c) 84 plain tests, no correction", "(d) 84 NW + HLN tests, Holm", "Context: 9 primary NW + HLN tests, Holm"]
labels = ["84 plain tests\nno correction", "84 NW + HLN\ntests, Holm", "9 primary tests\nNW + HLN, Holm"]
blocks = [5, 10, 20]
shades = [BLUE[250], BLUE[400], BLUE[700]]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharey=True)
for ax, n in zip(axes, [501, 441]):
    for k, (b, c) in enumerate(zip(blocks, shades)):
        sub = mc[(mc.n == n) & (mc.Mean_block == b)].set_index("Procedure").loc[procs]
        x = np.arange(len(procs)) + (k - 1) * 0.26
        ax.bar(x, sub.Rate_mean, width=0.24, color=c, label=f"mean block {b} days", edgecolor="white", linewidth=1)
        ax.errorbar(x, sub.Rate_mean, yerr=[sub.Rate_mean - sub.CI95_low, sub.CI95_high - sub.Rate_mean],
                    fmt="none", ecolor=INK2, elinewidth=0.7, capsize=1.5)
        if b == 10:
            for xi, v, top in zip(x, sub.Rate_mean, sub.CI95_high):
                ax.text(xi, top + 0.015, f"{v:.3f}", ha="center", fontsize=6.5, color=INK)
    ax.axhline(0.05, color=INK2, lw=0.8, ls=(0, (3, 2)))
    ax.text(-0.45, 0.058, "5% target", fontsize=6.5, color=INK2, ha="left", va="bottom")
    ax.set_xticks(range(len(procs)), labels, fontsize=7)
    ax.set_title(f"n = {n} test days", loc="left", fontsize=8)
    ax.yaxis.grid(True, color=GRID, lw=0.5)
axes[0].set_ylabel("P(at least one false winner)")
axes[0].set_ylim(0, 0.75)
axes[1].legend(loc="upper right", fontsize=6.5)
fig.text(0.01, 1.02, "No model is truly better (centred loss differentials, stationary block bootstrap); whiskers = 95% intervals",
         ha="left", fontsize=8, color=INK2)
save(fig, "Figure_3_monte_carlo_false_winners")

## CELL 7: Figure 4 - detection matrix (original protocol scope, seed S0)

In [ ]:
codes = {"pass": 0, "FAIL": 1, "FAIL (expected)": 2, "pass (MISSED)": 3}
txt = {"pass": "", "FAIL": "detected", "FAIL (expected)": "detected\n(expected)", "pass (MISSED)": "missed"}
cmap = ListedColormap([NEUTRAL, BLUE[250], BLUE[550], "#e34948"])
order = ["REF"] + FAULTS + ["COMBINED"]
M = matrix.loc[order]
vals = M.replace(codes).values.astype(int)
fig, ax = plt.subplots(figsize=(6.8, 3.6))
ax.imshow(vals, cmap=cmap, vmin=-0.5, vmax=3.5, aspect="auto")
for (i, j), v in np.ndenumerate(M.values):
    ax.text(j, i, txt[v], ha="center", va="center", fontsize=6.3, color="white" if v == "FAIL (expected)" else INK)
ax.set_xticks(range(7), [f"{c}\n{n}" for c, n in zip(CHECKS, ["Estimation\nlog", "Training\nfit", "Units", "Component\nidentity",
                                                             "Stage\ncontribution", "Forecast\norigin", "Evaluation\nspec"])], fontsize=6.5)
ylab = ["Reference (clean)"] + [f"{f} {FAULT_NAMES[f]}" + ("  *" if f in SINGLE_LINE else "") for f in FAULTS] + ["All eight combined"]
ax.set_yticks(range(len(order)), ylab, fontsize=7)
ax.set_xticks(np.arange(-0.5, 7), minor=True); ax.set_yticks(np.arange(-0.5, len(order)), minor=True)
ax.grid(which="minor", color="white", lw=2); ax.tick_params(which="minor", length=0, top=False)
ax.tick_params(axis="x", top=True, bottom=False, labeltop=True, labelbottom=False)
for s in ax.spines.values():
    s.set_visible(False)
fig.text(0.01, -0.02, "* detected by one check only (single line of defence). Coverage in this experiment (one pipeline, seed S0, "
         "faults injected one at a time); not a general reliability estimate. C2 on ARFIMA, LSTM and XGBoost (protocol scope).",
         fontsize=6.3, color=INK2, wrap=True)
save(fig, "Figure_4_detection_matrix")

## CELL 8: Figure 5 - architectures versus literature baselines, fair (reference) and v1 route (F8)

In [ ]:
bm = read("baselines/baseline_and_research_metrics_long.csv")
mean_ratio = bm.groupby(["Run", "Model"]).apply(lambda g: g.groupby("Seed").MAE_ratio.mean().mean()).unstack(0)
res = mean_ratio["research"].dropna().loc[["ARFIMA"] + ARCH]
ref_b, f8_b = mean_ratio["REF"].dropna().loc[BASELINES], mean_ratio["F8"].dropna().loc[BASELINES]
names = list(res.index) + BASELINES
fig, ax = plt.subplots(figsize=(6.8, 3.6))
y = np.arange(len(names))[::-1]
ax.axvline(1, color=INK2, lw=0.8)
ax.scatter(res.values, y[:len(res)], s=30, color=S1, edgecolor="white", lw=0.8, zorder=3, label="Research models (reference)")
yb = y[len(res):]
ax.hlines(yb, ref_b.values, f8_b.values, color=GRID, lw=1.5, zorder=1)
ax.scatter(ref_b.values, yb, s=30, color=S1, edgecolor="white", lw=0.8, zorder=3, marker="o")
ax.scatter(f8_b.values, yb, s=34, color=S2, edgecolor="white", lw=0.8, zorder=3, marker="D", label="Baselines, v1 route (F8)")
for yy, v in zip(yb, f8_b.values):
    ax.text(v * 1.12, yy, f"{v:.2f}", va="center", fontsize=6.5, color=INK2)
ax.set_xscale("log")
ax.set_yticks(y, names, fontsize=7)
ax.axhline(len(BASELINES) - 0.5, color=GRID, lw=0.8)
ax.text(0.98, len(BASELINES) - 0.42, "literature baselines below: dot = fair conditions, diamond = v1 route (F8)", transform=ax.get_yaxis_transform(), fontsize=6.5, color=INK2, ha="right", va="bottom")
ax.set_xlabel("MAE ratio to persistence, mean over stocks and seeds (log scale)")
ax.xaxis.grid(True, color=GRID, lw=0.5)
ticks = [t for t in [0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100] if ax.get_xlim()[0] <= t <= ax.get_xlim()[1]]
ax.set_xticks(ticks, [f"{t:g}" for t in ticks]); ax.minorticks_off()
ax.legend(loc="upper right", fontsize=6.5)
save(fig, "Figure_5_architectures_vs_baselines")

## CELL 9: Figure 6 - seed-to-seed spread of the reference (does a model beat persistence?)

In [ ]:
seeds = sorted(ref_m.Seed.unique())
fig, axes = plt.subplots(1, 3, figsize=(6.8, 3.0), sharey=True)
show = ["LSTM", "XGBoost", "Parallel SoftMax", "Parallel InvError", "Serial (Final)", "Stacking", "ARFIMA"]
for ax, t in zip(axes, TICKERS):
    sub = ref_m[(ref_m.Ticker == t) & ref_m.Model.isin(show)]
    for i, mname in enumerate(show[::-1]):
        v = sub[sub.Model == mname].MAE_ratio.values
        crosses = (v.min() < 1) and (v.max() > 1)
        ax.hlines(i, v.min(), v.max(), color=BLUE[250] if not crosses else S2, lw=2)
        ax.scatter(v, np.full(len(v), i), s=12, color=S1 if not crosses else S2, zorder=3, edgecolor="white", lw=0.5)
    ax.axvline(1, color=INK2, lw=0.8)
    ax.set_title(t, loc="left", fontsize=8)
    ax.xaxis.grid(True, color=GRID, lw=0.5)
    ax.tick_params(axis="x", labelsize=6.5)
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(3))
    ax.xaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter("%.3f"))
axes[0].set_yticks(range(len(show)), show[::-1], fontsize=7)
axes[1].set_xlabel("MAE ratio to persistence, five seeds (orange = crosses 1 depending on the seed alone)")
save(fig, "Figure_6_seed_spread")

## CELL 10: CHECKPOINT and run log

In [ ]:
figs = sorted(p.name for p in FIG.glob("*.png"))
tabs = sorted(p.name for p in TAB.glob("*.csv"))
checks = {"frozen_diagnostic_result_reproduced": REF_PASSES_ALL and ALL_DETECTED, "six_figures": len(figs) >= 6, "tables_written": len(tabs) >= 10}
print(pd.Series(checks).to_string())
print("Figures:", figs)
print("Tables:", tabs)
log = {"notebook": "ARTICLE2_NB5_Figures_and_Tables", "purpose": "Article 2 only", "run_at": datetime.now().isoformat(timespec="seconds"),
       "detected_by": detected_by, "single_line_of_defence": SINGLE_LINE, "figures": figs, "tables": tabs, "checks": checks}
with open(OUT / "logs" / "ARTICLE2_NB5_run_log.json", "w") as f:
    json.dump(log, f, indent=2)
assert all(checks.values()), "CHECKPOINT FAILED"
print()
print("Hard checks: ALL PASSED")
print("Please share the outputs of CELLS 2 and 10. The figures are in 05_Outputs/figures.")